# 15 · Private Credit & Funding Trades

Term loans, revolving credit facilities, and private markets funds share JSON-driven specs. This notebook loads sample contracts from the finstack test suite and prices them end-to-end.

### Learning Objectives
- Load JSON specs for term loans and revolving credit facilities
- Price each instrument with discount/forward curves
- Evaluate private markets funds and inspect LP cashflow ledgers

In [ ]:
from datetime import date
from pathlib import Path
import json
import sys

from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve, ForwardCurve
from finstack.valuations.instruments import TermLoan, RevolvingCredit, PrivateMarketsFund
from finstack.valuations.pricer import standard_registry

as_of = date(2024, 1, 2)
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9980), (1.0, 0.9960), (3.0, 0.9820), (5.0, 0.9550)],
    )
)
market.insert_forward(
    ForwardCurve(
        "USD-SOFR-3M",
        0.25,
        [(0.0, 0.05), (1.0, 0.0525), (3.0, 0.0550)],
        base_date=as_of,
    )
)
registry = standard_registry()

# Resolve the structured test fixtures from any ancestor that contains the Rust sources.
search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
attempted_paths = []
TEST_ROOT = None
for root in search_roots:
    candidate = root / "finstack" / "valuations" / "tests" / "instruments" / "json_examples"
    attempted_paths.append(candidate)
    if candidate.exists():
        TEST_ROOT = candidate
        break

if TEST_ROOT is None:
    attempted = [str(path) for path in attempted_paths]
    raise FileNotFoundError(
        "Could not find test data directory. "
        f"Tried: {attempted}"
    )

term_spec = json.loads((TEST_ROOT / "term_loan.json").read_text())["instrument"]["spec"]
rcf_spec = json.loads((TEST_ROOT / "revolving_credit.json").read_text())["instrument"]["spec"]
fund_spec = json.loads((TEST_ROOT / "private_markets_fund.json").read_text())["instrument"]["spec"]

term_loan = TermLoan.from_json(json.dumps(term_spec))
rcf = RevolvingCredit.from_json(json.dumps(rcf_spec))
fund = PrivateMarketsFund.from_json(json.dumps(fund_spec))


## 1. Term Loan
Term loans include amortization schedules, coupon specs, and optional delayed-draw features.

In [ ]:
term_res = registry.price(term_loan, "discounting", market, as_of=as_of)
print("Term loan PV:", term_res.value)


## 2. Revolving Credit Facility
RCFs reference SOFR-based spreads, draw/re-pay schedules, and fee tiers.

In [ ]:
rcf_res = registry.price(rcf, "discounting", market, as_of=as_of)
print("RCF PV:", rcf_res.value)


## 3. Private Markets Fund
Funds encode LP/GP waterfalls and contribution/distribution ledgers. Use helper methods to inspect LP cashflows.

In [ ]:
try:
    fund_res = registry.price(fund, "discounting", market, as_of=as_of)
    print("Fund PV:", fund_res.value)
except RuntimeError as e:
    if "Failed to solve for preferred return amount" in str(e):
        print("Note: Preferred return calculation failed (this can happen with complex waterfall structures).")
        print("Attempting to compute LP cashflows directly...")
        try:
            lp_flows = fund.lp_cashflows()
            print(f"LP cashflows computed: {len(lp_flows)} flows")
            print("First three LP cashflows:")
            for row in lp_flows[:3]:
                print("  ", row)
        except Exception as e2:
            print(f"Could not compute LP cashflows: {e2}")
    else:
        raise
else:
    print("First three LP cashflows:")
    for row in fund.lp_cashflows()[:3]:
        print("  ", row)


## Summary
- JSON specs from the finstack schema cover term loans, revolving credit facilities, and fund waterfalls.
- Pricing leverages the same `MarketContext` aggregator; only the JSON contract changes.
- Use instrument helpers (such as `lp_cashflows`) to expose bespoke analytics alongside PV.